# F1 Strategy Intelligence Suite: Strategy & Pit-Wall Simulation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neevoswal19-maker/f1-strategy-suite/blob/main/notebook/F1_Strategy_Intelligence_Suite.ipynb)

Models optimal pit windows and tyre strategies with **Monte Carlo simulation** and **ML lap-pace projections**, for past and upcoming races, dry or wet.

**Stack:** Python, NumPy, Pandas, Scikit-learn, FastF1, Ergast (Jolpica), OpenF1 API, Open-Meteo (rain forecast)

**How to run**
1. Pick a season, a race and (optionally) a driver in the form in **1.1**.
2. **Runtime → Run all.** Nothing asks for input after that.
3. To try another race, change the form and use **Runtime → Run all** again. Downloads are cached, so later runs are faster.

**Pipeline**

| Part | What happens |
|---|---|
| 1. Setup | your choices, installs, every setting in one place, helper functions |
| 2. The race | calendar, race data (with a dry-pace fallback for wet races), rain chance |
| 3. Tyre pace model | lap cleaning, degradation plot, Ridge vs Random Forest, pace projections |
| 4. Strategy search | pit loss, every legal 1- and 2-stop strategy ranked |
| 5. Monte Carlo | dry + wet race engine, top strategies, dry vs wet breakdown, pit windows |
| 6. Pit wall | "box now or stay out?" calls, wet tyre crossovers |
| 7. Real world | real strategies from OpenF1 vs the simulator's pick |
| 8. Race winner | the winner prediction lives on the project website |

# 1. Setup
### 1.1 Your choices
Fill in the form, then use **Runtime → Run all**.
- **SEASON**: any season from 2019 onwards.
- **RACE**: round number or Grand Prix name (`5`, `Monaco`, `Silverstone`). Leave it blank for the next race of that season (or its last race if the season is over).
- **DRIVER**: three-letter code (`VER`, `NOR`, `LEC`) whose pit strategy is simulated. Leave it blank to use the race winner.
- **SAVE_CACHE_TO_DRIVE**: keep downloaded data in Google Drive so it survives a Colab restart (Colab will ask for Drive access).

In [ ]:
#@title Race, driver and cache { display-mode: "form" }
SEASON = 2026  #@param {type:"integer"}
RACE = ""  #@param {type:"string"}
DRIVER = ""  #@param {type:"string"}
SAVE_CACHE_TO_DRIVE = False  #@param {type:"boolean"}

### 1.2 Install and import

In [ ]:
%pip install -q "fastf1>=3.9,<4"

import os, sys, time, itertools, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import requests_cache
import fastf1
from fastf1.ergast import Ergast

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, r2_score

warnings.filterwarnings('ignore')
fastf1.set_log_level('ERROR')

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and SAVE_CACHE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = '/content/drive/MyDrive/f1_cache'
elif IN_COLAB:
    CACHE_DIR = '/content/f1_cache'
else:
    CACHE_DIR = os.environ.get('F1_CACHE_DIR', os.path.abspath('f1_cache'))
os.makedirs(CACHE_DIR, exist_ok=True)
fastf1.Cache.enable_cache(CACHE_DIR)

# FastF1 3.9+ asks for an F1 TV login before live-timing downloads. This notebook doesn't need one
# (see 1.4), so turn the login prompt off unless you have set F1_USERNAME and F1_PASSWORD yourself.
try:
    from fastf1.internals.f1account import login as _f1_login
    from fastf1.exceptions import F1AuthError
    if not (os.environ.get('F1_USERNAME') and os.environ.get('F1_PASSWORD')):
        def _no_login():
            raise F1AuthError('no F1 account configured')
        _f1_login._credentials_from_env_or_prompt = _no_login
except ImportError:
    pass
print(f'FastF1 {fastf1.__version__} | cache: {CACHE_DIR}')

### 1.3 Settings
Every number the simulator assumes, in one place. Change any of them and **Run all** again.

In [ ]:
# --- simulation ---
N_SIMS = 2000            # Monte Carlo races per strategy (more = smoother, slower)
SEED   = 42
FUEL_K = 0.06            # seconds gained per lap as fuel burns off

# --- tyres ---
DRY    = ['SOFT', 'MEDIUM', 'HARD']
COLORS = {'SOFT': '#DA291C', 'MEDIUM': '#FFD12E', 'HARD': '#EBEBEB',
          'INTERMEDIATE': '#43B02A', 'WET': '#0067AD'}
MIN_COMPOUND_LAPS = 30   # clean laps a compound needs before we trust its pace curve

# --- strategy rules ---
MIN_STINT = 5            # shortest stint allowed (laps)
MAX_STOPS = 2            # most pit stops to consider

# --- pit stops and safety cars ---
DEFAULT_PIT_LOSS = 22.0  # used when the race has too few clean stops to measure it
P_SC          = 0.5      # chance of at least one safety car (raise to ~0.8 for street circuits)
SC_LAPS       = 4        # laps each safety car lasts
SC_SLOWDOWN   = 1.4      # safety-car lap = 1.4 x a normal lap
SC_PIT_FACTOR = 0.5      # a stop under the safety car costs about half
P_SLOW_STOP   = 0.05     # chance of a slow stop
SLOW_STOP_S   = 3.0      # extra time lost in a slow stop
DEG_UNCERT    = 0.15     # +/-15% uncertainty in tyre wear

# --- weather ---
RAIN_DEFAULT   = 0.15    # what-if rain chance when there is no forecast
WET_RACE_SHARE = 0.4     # above this share of wet-tyre laps, learn slick pace from another race
TYPES       = ['SLICK', 'INTERMEDIATE', 'WET']
GRIP        = {'SLICK': (1.00, 0.45), 'INTERMEDIATE': (1.06, 0.16), 'WET': (1.12, 0.04)}  # lap = dry pace x (a + b x wetness)
DRYING_LAPS = 8          # laps for the track to dry once the rain stops
LOOKAHEAD   = 5          # laps ahead the team considers when making a weather call

# --- pit windows ---
WINDOW_TOL  = 1.0        # pit laps within 1 s of the best count as the window
WINDOW_SPAN = 10         # how far either side of the ideal lap to test

# --- history ---
LOOKBACK_SEASONS = 3     # seasons searched for an earlier race at the same circuit

SITE_URL = 'https://neevoswal19-maker.github.io/f1-strategy-suite/'

### 1.4 Helper functions
**Where the lap data comes from.** Since 2026, FastF1 needs a paid F1 TV login for live-timing data from 2023 onwards. This notebook gets 2023+ laps, tyres, weather and results from the free **OpenF1 API**, and older races from FastF1's free archive. Both are turned into the same lap table, so the rest of the notebook doesn't care which one was used.

Circuits are matched by their Ergast `circuitId`, because FastF1 location names change between seasons (Miami → Miami Gardens, Yas Island → Yas Marina).

In [ ]:
from types import SimpleNamespace

erg = Ergast()

def season_schedule(year):
    """FastF1 calendar plus the Ergast circuit id and coordinates of each round."""
    s = fastf1.get_event_schedule(year, include_testing=False)
    try:
        e = erg.get_race_schedule(season=year).set_index('round')
        for col in ['circuitId', 'lat', 'long']:
            s[col] = s['RoundNumber'].map(e[col])
    except Exception:
        s['lat'] = s['long'] = np.nan
        s['circuitId'] = np.nan
    s['circuitId'] = s['circuitId'].fillna(s['Location'].str.lower())
    return s

def race_finished(s, now):
    return s['Session5DateUtc'] + pd.Timedelta(hours=3) < now

def past_races_at(circuit_id, before_year, seasons=LOOKBACK_SEASONS):
    """(year, round) of races at this circuit in the seasons before `before_year`, newest first."""
    out = []
    for y in range(before_year - 1, max(before_year - seasons - 1, 2017), -1):
        s = season_schedule(y)
        out += [(y, int(r)) for r in s.loc[s['circuitId'] == circuit_id, 'RoundNumber']][::-1]
    return out

# ---------------- OpenF1 ----------------
OPENF1 = 'https://api.openf1.org/v1'
http = requests_cache.CachedSession(os.path.join(CACHE_DIR, 'openf1_cache'), allowable_codes=(200,))

def openf1(endpoint, cached=True, **params):
    """GET an OpenF1 endpoint as a DataFrame. Throttled, retries on rate limits, 404 = empty."""
    get = http.get if cached else requests.get
    for attempt in range(6):
        r = get(f'{OPENF1}/{endpoint}', params=params, timeout=60)
        if r.status_code == 429:
            time.sleep(5 * (attempt + 1))
            continue
        if r.status_code == 404:
            return pd.DataFrame()
        r.raise_for_status()
        if not getattr(r, 'from_cache', False):
            time.sleep(0.4)
        data = r.json()
        return pd.DataFrame(data if isinstance(data, list) else [])
    raise RuntimeError('OpenF1 is rate-limiting. Wait a minute and run this cell again.')

def utc(ts):
    return pd.to_datetime(ts, utc=True, format='ISO8601').dt.tz_convert(None)

def neutralised_periods(rc):
    """(start, end) times of safety cars, virtual safety cars and red flags from race control."""
    periods, start = [], None
    for _, m in rc.sort_values('date').iterrows():
        msg, flag = str(m.get('message', '')).upper(), str(m.get('flag', '')).upper()
        if start is None and ('DEPLOYED' in msg or flag == 'RED'):
            start = m['date']
        elif start is not None and ('ENDING' in msg or 'IN THIS LAP' in msg or
                                    (flag in ('GREEN', 'CLEAR') and str(m.get('scope', '')) == 'Track')):
            periods.append((start, m['date'] + pd.Timedelta(seconds=90)))
            start = None
    if start is not None:
        periods.append((start, rc['date'].max()))
    return periods

def race_from_openf1(year, rnd, race_utc, name):
    races = openf1('sessions', year=year, session_name='Race')
    if races.empty:
        return None
    races['start'] = utc(races['date_start'])
    gap = (races['start'] - race_utc).abs()
    if gap.min() > pd.Timedelta(days=2):
        return None
    key = int(races.loc[gap.idxmin(), 'session_key'])
    cached = race_utc < NOW - pd.Timedelta(days=2)
    get = lambda ep: openf1(ep, cached=cached, session_key=key)

    l, st, drv, wx, rc, res, pit = (get(ep) for ep in
        ['laps', 'stints', 'drivers', 'weather', 'race_control', 'session_result', 'pit'])
    if l.empty or st.empty:
        return None
    code = drv.drop_duplicates('driver_number').set_index('driver_number')['name_acronym']

    l = l.dropna(subset=['lap_duration']).copy()
    l['LapStart'] = utc(l['date_start'])
    st = st.dropna(subset=['lap_start']).copy()
    st['lap_end'] = st['lap_end'].fillna(l['lap_number'].max())
    st['tyre_age_at_start'] = st['tyre_age_at_start'].fillna(0)
    tyres = [(r['driver_number'], lap, r['compound'], r['tyre_age_at_start'] + lap - r['lap_start'] + 1)
             for _, r in st.iterrows() for lap in range(int(r['lap_start']), int(r['lap_end']) + 1)]
    tyres = pd.DataFrame(tyres, columns=['driver_number', 'lap_number', 'Compound', 'TyreLife'])
    tyres = tyres.drop_duplicates(['driver_number', 'lap_number'], keep='last')
    l = l.merge(tyres, on=['driver_number', 'lap_number'], how='left')

    pit_in = set(zip(pit['driver_number'], pit['lap_number'])) if len(pit) else set()
    periods = []
    if len(rc):
        rc['date'] = utc(rc['date'])
        periods = neutralised_periods(rc)
    end = l['LapStart'] + pd.to_timedelta(l['lap_duration'], unit='s')
    green = pd.Series(True, index=l.index)
    for a, b in periods:
        green &= ~((l['LapStart'] < b) & (end > a))

    laps = pd.DataFrame({
        'Driver': l['driver_number'].map(code), 'LapNumber': l['lap_number'],
        'LapTimeS': l['lap_duration'], 'Compound': l['Compound'].str.upper(), 'TyreLife': l['TyreLife'],
        'PitIn': [(d, n) in pit_in for d, n in zip(l['driver_number'], l['lap_number'])],
        'PitOut': l['is_pit_out_lap'].fillna(False).astype(bool), 'Green': green, 'LapStart': l['LapStart']})
    weather = pd.DataFrame({'Time': utc(wx['date']), 'TrackTemp': wx['track_temperature'],
                            'Rainfall': wx['rainfall'].fillna(0) > 0}) if len(wx) else \
              pd.DataFrame({'Time': [laps['LapStart'].min()], 'TrackTemp': [35.0], 'Rainfall': [False]})
    results = pd.DataFrame({'Abbreviation': res['driver_number'].map(code), 'Position': res['position']}) \
              if len(res) else laps.groupby('Driver')['LapNumber'].max().sort_values(ascending=False) \
                                 .rank(ascending=False, method='first').rename('Position').reset_index() \
                                 .rename(columns={'Driver': 'Abbreviation'})
    total = int(res['number_of_laps'].max()) if len(res) and 'number_of_laps' in res else int(laps['LapNumber'].max())
    return SimpleNamespace(year=year, round=rnd, name=name, date=race_utc, laps=laps,
                           weather=weather, results=results.dropna(subset=['Abbreviation']),
                           total_laps=total, source=f'OpenF1 session {key}')

# ---------------- FastF1 archive (2018-2022) ----------------
def race_from_fastf1(year, rnd):
    s = fastf1.get_session(year, rnd, 'R')
    s.load(laps=True, telemetry=False, weather=True, messages=False)
    l = s.laps
    if not len(l):
        return None
    t0 = pd.Timestamp('2000-01-01')   # laps and weather share the session clock; any fixed base works
    laps = pd.DataFrame({
        'Driver': l['Driver'], 'LapNumber': l['LapNumber'], 'LapTimeS': l['LapTime'].dt.total_seconds(),
        'Compound': l['Compound'], 'TyreLife': l['TyreLife'],
        'PitIn': l['PitInTime'].notna(), 'PitOut': l['PitOutTime'].notna(),
        'Green': l['TrackStatus'] == '1', 'LapStart': t0 + l['LapStartTime']})
    try:
        w = s.weather_data
        weather = pd.DataFrame({'Time': t0 + w['Time'], 'TrackTemp': w['TrackTemp'], 'Rainfall': w['Rainfall']})
    except Exception:          # the archive has no weather for some races
        weather = pd.DataFrame({'Time': [laps['LapStart'].min()], 'TrackTemp': [35.0], 'Rainfall': [False]})
    try:
        total = int(s.total_laps)
    except Exception:          # FastF1 raises its own error when the lap count is missing
        total = int(laps['LapNumber'].max())
    return SimpleNamespace(year=year, round=rnd, name=s.event['EventName'], date=pd.Timestamp(s.date),
                           laps=laps.dropna(subset=['LapTimeS']), weather=weather,
                           results=s.results[['Abbreviation', 'Position']].copy(),
                           total_laps=total, source='FastF1 archive')

def load_race(year, rnd):
    """One race as a standard lap table (OpenF1 for 2023+, FastF1 before), or None if there is no data."""
    try:
        ev = season_schedule(year).get_event_by_round(rnd)
        if year >= 2023:
            return race_from_openf1(year, rnd, ev['Session5DateUtc'], ev['EventName'])
        return race_from_fastf1(year, rnd)
    except Exception as err:
        print(f'   could not load {year} round {rnd}: {err}')
        return None

def wet_share(l):
    return l['Compound'].isin(['INTERMEDIATE', 'WET']).mean()

# ---------------- weather forecast ----------------
def rain_forecast(lat, lon, start_utc, hours=2):
    """Highest hourly rain probability (0-1) from the race start to `hours` later. Open-Meteo, 16 days ahead."""
    fc = requests.get('https://api.open-meteo.com/v1/forecast',
                      params={'latitude': lat, 'longitude': lon, 'forecast_days': 16,
                              'hourly': 'precipitation_probability', 'timezone': 'GMT'}, timeout=20).json()
    hourly = pd.DataFrame(fc['hourly'])
    hourly['time'] = pd.to_datetime(hourly['time'])
    window = hourly[(hourly['time'] >= start_utc.floor('h')) &
                    (hourly['time'] <= start_utc + pd.Timedelta(hours=hours))]
    p = window['precipitation_probability'].max()
    return None if pd.isna(p) else p / 100

def geocode(place):
    g = requests.get('https://geocoding-api.open-meteo.com/v1/search',
                     params={'name': place, 'count': 1}, timeout=20).json()
    return g['results'][0]['latitude'], g['results'][0]['longitude']

# 2. The race
### 2.1 Calendar and race picker
Shows the season, marks each round `done` or `UPCOMING`, and works out which real race supplies the data. An upcoming race is projected from the last race held at the same circuit. A brand-new circuit has no earlier race to learn from, so the notebook switches to the latest completed race and says so.

In [ ]:
NOW = pd.Timestamp.now(tz='UTC').tz_localize(None)

YEAR = int(SEASON)
schedule = season_schedule(YEAR)
if schedule.empty:
    print(f'No calendar for {YEAR} yet, using {NOW.year} instead.')
    YEAR = NOW.year
    schedule = season_schedule(YEAR)
finished = race_finished(schedule, NOW)

seen = set()
for y in range(YEAR - 1, YEAR - LOOKBACK_SEASONS - 1, -1):
    seen |= set(season_schedule(y)['circuitId'])
show = schedule[['RoundNumber', 'EventName', 'Location']].copy()
show['Date']   = schedule['EventDate'].dt.strftime('%d %b')
show['Format'] = np.where(schedule['EventFormat'].str.contains('sprint'), 'sprint', '')
show['Status'] = np.where(finished, 'done',
                          np.where(schedule['circuitId'].isin(seen), 'UPCOMING', 'UPCOMING (new circuit)'))
print(f'{YEAR} calendar')
print(show.to_string(index=False))

def pick_event(choice):
    choice = str(choice).strip()
    try:
        if choice.isdigit():
            return schedule.get_event_by_round(int(choice))
        if choice:
            return schedule.get_event_by_name(choice)
    except Exception:
        print(f'\nCould not find "{choice}" in the {YEAR} calendar, so I picked the next race.')
    ahead = schedule[~finished]
    row = ahead.iloc[0] if len(ahead) else schedule.iloc[-1]
    return schedule.get_event_by_round(int(row['RoundNumber']))

event   = pick_event(RACE)
CIRCUIT = event['circuitId']
race_utc = event['Session5DateUtc']
UPCOMING = race_utc + pd.Timedelta(hours=3) >= NOW
DATA_CANDIDATES = []      # (year, round) races that can supply data, best first

if UPCOMING:
    days = max((race_utc - NOW).days, 0)
    print(f"\n>>> UPCOMING RACE: {event['EventName']} {YEAR} on {race_utc:%d %b %Y} ({days} days to go)")
    DATA_CANDIDATES = past_races_at(CIRCUIT, YEAR)
    if DATA_CANDIDATES:
        print(f"    No race data exists yet, so I will project it from the last race at {event['Location']}.")
    else:
        done = schedule[finished]
        if len(done):
            latest = schedule.get_event_by_round(int(done.iloc[-1]['RoundNumber']))
        else:
            prev = season_schedule(YEAR - 1)
            YEAR, schedule = YEAR - 1, prev
            latest = prev.get_event_by_round(int(prev.iloc[-1]['RoundNumber']))
        print(f"    {event['Location']} is a new circuit with no earlier race to learn from.")
        print(f"    Simulating the latest completed race instead: {latest['EventName']} {YEAR}.")
        event, CIRCUIT, UPCOMING = latest, latest['circuitId'], False
        race_utc = event['Session5DateUtc']
if not UPCOMING:
    print(f"\n>>> COMPLETED RACE: {event['EventName']} {YEAR} ({race_utc:%d %b %Y})")
    DATA_CANDIDATES = [(YEAR, int(event['RoundNumber']))] + past_races_at(CIRCUIT, YEAR)

### 2.2 Load race data and pick the strategy driver
Loads the race. If more than 40% of laps were on wet tyres, slick pace is learned from an earlier dry race at the same circuit (or, failing that, from the slick laps of this race).

In [ ]:
race = None
for DATA_YEAR, DATA_ROUND in DATA_CANDIDATES:
    race = load_race(DATA_YEAR, DATA_ROUND)
    if race is not None:
        break
if race is None:   # nothing usable at this circuit: fall back to the latest race with data
    fallback_years = [YEAR, YEAR - 1] + [y for y in (NOW.year, NOW.year - 1) if y >= 2023 and y < YEAR - 1 or y > YEAR]
    for y in dict.fromkeys(fallback_years):
        s = season_schedule(y)
        for r in list(s.loc[race_finished(s, NOW), 'RoundNumber'][::-1])[:3]:   # try the 3 latest races per season
            race = load_race(y, int(r))
            if race is not None:
                DATA_YEAR, DATA_ROUND = y, int(r)
                CIRCUIT = s.loc[s['RoundNumber'] == r, 'circuitId'].iloc[0]
                print(f'No lap data at this circuit, using {race.name} {y} instead.')
                event, UPCOMING, YEAR = s.get_event_by_round(int(r)), False, y
                break
        if race is not None:
            break

race_laps  = race.laps
TOTAL_LAPS = race.total_laps
WAS_WET    = bool(race.weather['Rainfall'].any()) or wet_share(race_laps) > 0
RACE_TITLE = f"{event['EventName']} {YEAR}" + (f' (projected from {DATA_YEAR})' if UPCOMING else '')

session, laps = race, race_laps                  # the race whose laps teach us dry tyre pace
if wet_share(race_laps) > WET_RACE_SHARE:
    print(f'{wet_share(race_laps):.0%} of laps were on wet tyres, looking for a dry race here to learn slick pace')
    for y, r in past_races_at(CIRCUIT, DATA_YEAR):
        s = load_race(y, r)
        if s is not None and wet_share(s.laps) < WET_RACE_SHARE:
            session, laps = s, s.laps
            print('   dry pace will come from the', y, 'race')
            break
    else:
        print('   no dry race here in the last seasons, so slick pace comes from the slick laps of this race')

# strategy driver: your choice, else the winner of the race the data comes from
winner = race.results.sort_values('Position')['Abbreviation'].iloc[0]
DRIVER_CODE = str(DRIVER).strip().upper() or winner
if DRIVER_CODE not in set(laps['Driver']):
    alt = session.results.sort_values('Position')['Abbreviation'].iloc[0]
    print(f'{DRIVER_CODE} has no laps in the {session.year} race used for tyre pace, so I am using {alt} instead')
    DRIVER_CODE = alt

print(f'\nSimulating: {RACE_TITLE}')
print(f'Data race: {race.name} {DATA_YEAR} ({race.source}) | laps: {TOTAL_LAPS} | wet: {WAS_WET} | '
      f'pace from: {session.name} {session.year}')
print(f'Strategy driver: {DRIVER_CODE}')
print(race_laps['Compound'].value_counts().to_string())

### 2.3 Rain chance: forecast or what-if
Upcoming race: Open-Meteo forecast for the race window (16 days ahead at most). Past wet race: rain in every simulated race. Past dry race: a what-if `RAIN_DEFAULT`.

In [ ]:
if UPCOMING:
    P_RAIN = None
    try:
        lat, lon = float(event['lat']), float(event['long'])
        if np.isnan(lat):
            lat, lon = geocode(event['Location'])
        P_RAIN = rain_forecast(lat, lon, race_utc)
    except Exception:
        P_RAIN = None
    if P_RAIN is None:
        P_RAIN = RAIN_DEFAULT
        print(f'No forecast yet (race more than 16 days away or lookup failed). Using a {P_RAIN:.0%} rain chance.')
    else:
        print(f"Forecast: {P_RAIN:.0%} chance of rain during the race at {event['Location']}")
elif WAS_WET:
    P_RAIN = 1.0
    print('This race was WET in real life, so every simulated race gets rain.')
else:
    P_RAIN = RAIN_DEFAULT
    print(f'This race was dry. Simulating a {P_RAIN:.0%} what-if chance of rain.')

# 3. Tyre pace model
### 3.1 Clean laps, fuel correction, track temperature
Keeps green-flag laps without pit stops, within 107% of the fastest lap, on dry compounds, and drops lap 1.

In [ ]:
fastest = laps['LapTimeS'].min()
clean = laps[~laps['PitIn'] & ~laps['PitOut'] & laps['Green'] &
             (laps['LapTimeS'] <= 1.07 * fastest) & (laps['LapNumber'] > 1) &
             laps['Compound'].isin(DRY)]
clean = clean.dropna(subset=['LapTimeS', 'TyreLife', 'LapStart']).copy()

# remove the effect of fuel weight so only tyre wear is left
clean['FuelCorrS'] = clean['LapTimeS'] - FUEL_K * (TOTAL_LAPS - clean['LapNumber'])

# attach track temperature at the start of each lap
weather = session.weather[['Time', 'TrackTemp']].dropna().sort_values('Time')
clean = pd.merge_asof(clean.sort_values('LapStart'), weather,
                      left_on='LapStart', right_on='Time', direction='nearest')
clean['TrackTemp'] = clean['TrackTemp'].fillna(weather['TrackTemp'].median())

print(len(clean), 'clean green-flag laps')
print(clean['Compound'].value_counts().to_string())

### 3.2 Exploratory plot: tyre degradation

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for c in DRY:
    d = clean[clean['Compound'] == c]
    if len(d):
        ax.scatter(d['TyreLife'], d['FuelCorrS'], s=10, alpha=0.4,
                   color=COLORS[c], edgecolors='k', linewidths=0.2, label=c)
ax.set(xlabel='Tyre age (laps)', ylabel='Fuel-corrected lap time (s)',
       title=f'Tyre degradation by compound - {session.name} {session.year}')
ax.legend()
plt.show()

### 3.3 Feature engineering and pace models
Ridge with per-compound tyre age and age² terms, track temperature and driver dummies, against a Random Forest (5-fold CV). Ridge is kept: tree models go flat beyond the tyre ages they have seen, which would make long stints look free.

In [ ]:
COMPOUNDS = sorted(clean['Compound'].unique())

def make_features(df, columns=None):
    X = pd.DataFrame(index=df.index)
    for c in COMPOUNDS:
        is_c = (df['Compound'] == c).astype(float)
        X[f'is_{c}']   = is_c
        X[f'age_{c}']  = is_c * df['TyreLife']
        X[f'age2_{c}'] = is_c * df['TyreLife'] ** 2
    X['TrackTemp'] = df['TrackTemp']
    X = pd.concat([X, pd.get_dummies(df['Driver'], prefix='drv', dtype=float)], axis=1)
    if columns is not None:
        X = X.reindex(columns=columns, fill_value=0.0)
    return X

X = make_features(clean)
y = clean['FuelCorrS']
FEATURES = X.columns

models = {
    'Ridge':        make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    'RandomForest': RandomForestRegressor(n_estimators=300, min_samples_leaf=5,
                                          random_state=SEED, n_jobs=-1),
}
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
cv_pred = {}
for name, m in models.items():
    cv_pred[name] = cross_val_predict(m, X, y, cv=cv)
    print(f'{name:13s} MAE = {mean_absolute_error(y, cv_pred[name]):.3f} s'
          f'   R2 = {r2_score(y, cv_pred[name]):.3f}')

pace_model = models['Ridge'].fit(X, y)
LAP_NOISE  = np.std(y - cv_pred['Ridge'])
print(f'Lap-to-lap noise: {LAP_NOISE:.2f} s')

### 3.4 Lap pace projections per compound
Pace curve per compound for the strategy driver, kept non-decreasing with tyre age. A compound needs `MIN_COMPOUND_LAPS` clean laps to be used; the longest stint allowed is 1.1 × the longest one seen.

In [ ]:
MAX_AGE    = TOTAL_LAPS + 5
TRACK_TEMP = clean['TrackTemp'].median()
counts     = clean['Compound'].value_counts()

AVAILABLE = [c for c in COMPOUNDS if counts[c] >= MIN_COMPOUND_LAPS]
if len(AVAILABLE) < 2:
    AVAILABLE = [c for c in COMPOUNDS if counts[c] >= 10] or [counts.idxmax()]
TWO_COMPOUND_RULE = len(AVAILABLE) >= 2
if not TWO_COMPOUND_RULE:
    print(f'Only {AVAILABLE[0]} has enough clean laps, so the two-compound rule is switched off for this race.')

MAX_STINT = {c: max(min(int(clean.loc[clean['Compound'] == c, 'TyreLife'].max() * 1.1), TOTAL_LAPS), MIN_STINT)
             for c in AVAILABLE}

ages = np.arange(1, MAX_AGE + 1)
PACE = {}
for c in AVAILABLE:
    grid = pd.DataFrame({'Driver': DRIVER_CODE, 'Compound': c,
                         'TyreLife': ages, 'TrackTemp': TRACK_TEMP})
    PACE[c] = np.maximum.accumulate(pace_model.predict(make_features(grid, FEATURES)))

CUM = {c: np.concatenate([[0.0], np.cumsum(PACE[c])]) for c in PACE}

deg_table = pd.DataFrame({
    'fresh_pace_s':  {c: PACE[c][0] for c in AVAILABLE},
    'deg_s_per_lap': {c: (PACE[c][MAX_STINT[c] - 1] - PACE[c][0]) / max(MAX_STINT[c] - 1, 1) for c in AVAILABLE},
    'max_stint':     MAX_STINT,
}).round(3)
print(deg_table.to_string())

fig, ax = plt.subplots(figsize=(10, 5))
for c in AVAILABLE:
    d = clean[clean['Compound'] == c]
    ax.scatter(d['TyreLife'], d['FuelCorrS'], s=8, alpha=0.25,
               color=COLORS[c], edgecolors='k', linewidths=0.2)
    ax.plot(ages[:MAX_STINT[c]], PACE[c][:MAX_STINT[c]], color=COLORS[c], lw=3,
            label=f'{c} (model)')
ax.set(xlabel='Tyre age (laps)', ylabel='Fuel-corrected lap time (s)',
       title=f'Projected lap pace for {DRIVER_CODE} - {RACE_TITLE}')
ax.legend()
plt.show()

# 4. Strategy search
### 4.1 Pit-loss estimate
Pit loss = in-lap + out-lap minus twice the driver's median green lap, over every green-flag stop in the race. Safety-car and slow-stop settings are in 1.3.

In [ ]:
def estimate_pit_loss(laps):
    losses = []
    for drv, d in laps.groupby('Driver'):
        d = d.set_index('LapNumber')
        d = d[~d.index.duplicated()]
        ref = d.loc[d['Green'] & ~d['PitIn'] & ~d['PitOut'], 'LapTimeS'].median()
        for lap in d.index[d['PitIn']]:
            if lap + 1 in d.index and d.loc[lap + 1, 'PitOut']:
                pair = d.loc[[lap, lap + 1]]
                if pair['Green'].all():
                    losses.append(pair['LapTimeS'].sum() - 2 * ref)
    losses = pd.Series(losses, dtype=float)
    losses = losses[(losses > 10) & (losses < 40)]
    if len(losses) < 3:
        return DEFAULT_PIT_LOSS, 1.0
    return losses.median(), min(losses.std(), 2.0)

PIT_LOSS, PIT_SD = estimate_pit_loss(laps)
SC_LAP_TIME = SC_SLOWDOWN * clean['LapTimeS'].median()
print(f'Pit loss: {PIT_LOSS:.1f} s  (sd {PIT_SD:.1f} s) | safety-car lap: {SC_LAP_TIME:.1f} s')

### 4.2 Generate and rank every legal strategy
Every 1- and 2-stop plan with at least two dry compounds, stints of `MIN_STINT` laps or more and no stint longer than its compound allows. A quick deterministic ranking picks the best pit laps for each compound sequence; Monte Carlo then runs on those.

In [ ]:
def generate_strategies(min_stint=MIN_STINT, max_stops=MAX_STOPS):
    out = []
    for stops in range(1, max_stops + 1):
        for comps in itertools.product(AVAILABLE, repeat=stops + 1):
            if TWO_COMPOUND_RULE and len(set(comps)) < 2:   # rule: two different dry compounds
                continue
            for pits in itertools.combinations(range(min_stint, TOTAL_LAPS - min_stint + 1), stops):
                lengths = np.diff([0, *pits, TOTAL_LAPS])
                if lengths.min() < min_stint:
                    continue
                if any(n > MAX_STINT[c] for c, n in zip(comps, lengths)):
                    continue
                out.append(list(zip(comps, lengths)))
    return out

def stint_time(c, n, start_age=0):
    return CUM[c][start_age + n] - CUM[c][start_age]

def race_time(strategy):
    return sum(stint_time(c, n) for c, n in strategy) + PIT_LOSS * (len(strategy) - 1)

def label(strategy):
    pits = np.cumsum([n for _, n in strategy])[:-1]
    return '-'.join(c[0] for c, _ in strategy) + ' | box L' + ', L'.join(map(str, pits))

strategies = generate_strategies()
while not strategies and min(MAX_STINT.values()) < TOTAL_LAPS:
    MAX_STINT = {c: min(int(n * 1.25) + 1, TOTAL_LAPS) for c, n in MAX_STINT.items()}
    print('No plan fits the observed stint lengths, stretching max stints to', MAX_STINT)
    strategies = generate_strategies()

det = pd.DataFrame({'strategy': strategies})
det['time']  = det['strategy'].apply(race_time)
det['seq']   = det['strategy'].apply(lambda s: '-'.join(c[0] for c, _ in s))
det['label'] = det['strategy'].apply(label)

best_per_seq = det.loc[det.groupby('seq')['time'].idxmin()].sort_values('time')
best_per_seq['gap_s'] = best_per_seq['time'] - best_per_seq['time'].min()

print(f'{len(det):,} strategies evaluated')
print(best_per_seq[['label', 'gap_s']].head(10).round(2).to_string(index=False))

# 5. Monte Carlo
### 5.1 Monte Carlo engine (dry + wet)
Every strategy faces the same random races (lap noise, safety cars, wear uncertainty, pit-stop noise, slow stops, rain), so differences between strategies are low-noise. Dry races run vectorised; wet races run lap by lap with reactive weather calls. With no rain the wet engine gives exactly the dry result.

In [ ]:
FUEL    = FUEL_K * (TOTAL_LAPS - np.arange(1, TOTAL_LAPS + 1))
DRY_REF = min(PACE[c][0] for c in AVAILABLE)

def kind(tyre):
    return 0 if tyre in DRY else TYPES.index(tyre)

def mult(k, w):
    a, b = GRIP[TYPES[k]]
    return a + b * w

def best_slick(R):
    ok = [c for c in AVAILABLE if MAX_STINT[c] >= R]
    return min(ok, key=lambda c: stint_time(c, R)) if ok else max(AVAILABLE, key=MAX_STINT.get)

def make_scenarios(n=N_SIMS, seed=SEED):
    rng = np.random.default_rng(seed)
    t = np.arange(TOTAL_LAPS)[None, :]

    rain  = rng.random(n) < P_RAIN
    onset = rng.integers(1, max(TOTAL_LAPS - 5, 2), n)[:, None]
    dur   = rng.integers(5, 25, n)[:, None]
    peak  = rng.uniform(0.15, 1.0, n)[:, None]
    rise  = np.clip((t - onset + 1) / 2, 0, 1)
    dry   = np.clip(1 - (t - onset - dur) / DRYING_LAPS, 0, 1)
    wet   = peak * np.minimum(rise, dry) * rain[:, None]

    hazard = 1 - (1 - P_SC) ** (1 / TOTAL_LAPS)
    start = rng.random((n, TOTAL_LAPS)) < hazard * (1 + 5 * wet)   # rain = more safety cars
    sc = np.zeros_like(start)
    for k in range(SC_LAPS):
        sc[:, k:] |= start[:, :TOTAL_LAPS - k]

    M = np.stack([mult(k, wet) for k in range(3)])
    return {
        'rain': rain, 'wet': wet, 'mult': M, 'best': M.argmin(axis=0), 'sc': sc,
        'noise':     rng.normal(0, LAP_NOISE, (n, TOTAL_LAPS)),
        'deg_mult':  rng.normal(1, DEG_UNCERT, n).clip(0.5, 1.5),
        'pit_noise': rng.normal(0, PIT_SD, (n, TOTAL_LAPS)),
        'slow_stop': rng.random((n, TOTAL_LAPS)) < P_SLOW_STOP,
    }

def lap_profile(strategy):
    base, deg = [], []
    for c, n in strategy:
        p = PACE[c][:n]
        base.append(np.full(n, p[0]))
        deg.append(p - p[0])
    pit_laps = np.cumsum([n for _, n in strategy])[:-1] - 1
    return np.concatenate(base), np.concatenate(deg), pit_laps

def dry_races(strategy, S, idx):
    base, deg, pit_laps = lap_profile(strategy)
    sc = S['sc'][idx]
    lap_t = base + FUEL + deg * S['deg_mult'][idx][:, None] + S['noise'][idx]
    lap_t = np.where(sc, SC_LAP_TIME, lap_t)
    total = lap_t.sum(axis=1)
    for p in pit_laps:
        loss = PIT_LOSS + S['pit_noise'][idx, p] + SLOW_STOP_S * S['slow_stop'][idx, p]
        total += np.where(sc[:, p], loss * SC_PIT_FACTOR, loss)
    return total

def wet_race(strategy, S, i, trace=None):
    plan = [c for c, _ in strategy]
    pits = {int(p) for p in np.cumsum([n for _, n in strategy])[:-1]}
    tyre, age, stint, on_plan, total = plan[0], 0, 0, True, 0.0
    M, best = S['mult'][:, i], S['best'][i]

    for t in range(TOTAL_LAPS):
        lap, k, sc = t + 1, kind(tyre), S['sc'][i, t]
        dry_lap = DRY_REF + FUEL[t]
        if tyre in DRY:
            p = PACE[tyre]
            dry_lap = p[0] + (p[min(age, len(p) - 1)] - p[0]) * S['deg_mult'][i] + FUEL[t]
        total += SC_LAP_TIME if sc else dry_lap * M[k, t] + S['noise'][i, t]
        if trace is not None:
            trace.append(tyre)
        age += 1
        left = TOTAL_LAPS - lap
        if left == 0:
            break

        factor = SC_PIT_FACTOR if sc else 1.0
        b = best[t]
        gain = (DRY_REF + FUEL[t]) * (M[k, t] - M[b, t]) * min(left, LOOKAHEAD)
        new = None
        if k != b and gain > PIT_LOSS * factor:                         # weather call
            new = best_slick(left) if b == 0 else TYPES[b]
            on_plan = False
        elif on_plan and lap in pits:                                   # planned dry stop
            stint += 1
            new = plan[stint]
        elif tyre in DRY and not on_plan and age >= MAX_STINT[tyre]:    # slicks worn out
            new = best_slick(left)

        if new:
            total += (PIT_LOSS + S['pit_noise'][i, t] + SLOW_STOP_S * S['slow_stop'][i, t]) * factor
            tyre, age = new, 0
    return total

def monte_carlo(strategy, S):
    total = np.empty(len(S['rain']))
    dry = ~S['rain']
    if dry.any():
        total[dry] = dry_races(strategy, S, dry)
    for i in np.flatnonzero(S['rain']):
        total[i] = wet_race(strategy, S, i)
    return total

S = make_scenarios()
print(f"Simulated races with rain: {S['rain'].mean():.0%} | with a safety car: {S['sc'].any(axis=1).mean():.0%}")

### 5.2 Simulate the top strategies
The best plan for each of the top 6 compound sequences, raced `N_SIMS` times. `regret_p95_s` is the time lost to the best plan in the worst 5% of races; `p_fastest` is how often the plan was the quickest one.

In [ ]:
top  = best_per_seq.head(6).reset_index(drop=True)
sims = np.vstack([monte_carlo(s, S) for s in top['strategy']])

order = np.argsort(sims.mean(axis=1))
top, sims = top.iloc[order].reset_index(drop=True), sims[order]

regret  = sims - sims.min(axis=0)
fastest = np.bincount(sims.argmin(axis=0), minlength=len(top)) / N_SIMS

mc = pd.DataFrame({
    'strategy':     top['label'],
    'mean_gap_s':   sims.mean(axis=1) - sims.mean(axis=1).min(),
    'regret_p95_s': np.percentile(regret, 95, axis=1),
    'p_fastest':    fastest,
}).round(2)
print(mc.to_string(index=False))

BEST = top.loc[0, 'strategy']
print(f"\nRECOMMENDED for {DRIVER_CODE} at {RACE_TITLE}: {top.loc[0, 'label']}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].boxplot(regret.T, showfliers=False)
axes[0].set_xticks(range(1, len(top) + 1), top['label'], rotation=30, ha='right')
axes[0].set(ylabel='Time lost vs best plan in that race (s)', title='Strategy risk')
axes[1].barh(top['label'][::-1], fastest[::-1], color='tab:red')
axes[1].set(xlabel='Share of races where this plan was fastest', title=f'Fastest plan over {N_SIMS} simulated races')
plt.tight_layout()
plt.show()

### 5.3 Dry vs wet breakdown and a wet race replay

In [ ]:
rain = S['rain']
for name, mask in [('DRY races', ~rain), ('WET races', rain)]:
    if mask.sum() == 0:
        print(f'\n{name}: none in this run')
        continue
    part = sims[:, mask]
    print(f'\n{name}: {mask.sum()} simulations')
    print(pd.DataFrame({
        'strategy':   top['label'],
        'mean_gap_s': part.mean(axis=1) - part.mean(axis=1).min(),
        'p_fastest':  np.bincount(part.argmin(axis=0), minlength=len(top)) / mask.sum(),
    }).round(2).to_string(index=False))

heavy = np.flatnonzero(rain & (S['wet'].max(axis=1) > 0.5))
if len(heavy):
    i, trace = heavy[0], []
    wet_race(BEST, S, i, trace)
    fig, ax = plt.subplots(figsize=(12, 3.5))
    ax.plot(np.arange(TOTAL_LAPS) + 0.5, S['wet'][i], color='tab:blue', lw=2, label='Track wetness')
    for lap, tyre in enumerate(trace, 1):
        ax.axvspan(lap - 1, lap, ymax=0.1, color=COLORS[tyre])
    ax.set(xlabel='Lap', ylabel='Wetness', ylim=(0, 1.05),
           title=f'One simulated wet race: tyre used each lap (bottom strip) - {RACE_TITLE}')
    ax.legend()
    plt.show()
else:
    print('\nNo heavy-rain race in this run to replay (raise RAIN_DEFAULT in 1.3 to see one).')

### 5.4 Optimal pit windows
For each stop of the recommended plan, moves that pit lap up to `WINDOW_SPAN` laps either way and takes the Monte Carlo mean. The window is every lap within `WINDOW_TOL` seconds of the best.

In [ ]:
def with_pits(comps, pits):
    return list(zip(comps, np.diff([0, *pits, TOTAL_LAPS])))

def feasible(strategy, min_stint=MIN_STINT):
    return all(min_stint <= n <= MAX_STINT[c] for c, n in strategy)

comps     = [c for c, _ in BEST]
best_pits = [int(p) for p in np.cumsum([n for _, n in BEST])[:-1]]

fig, ax = plt.subplots(figsize=(10, 5))
for i, p0 in enumerate(best_pits):
    xs, ys = [], []
    for p in range(p0 - WINDOW_SPAN, p0 + WINDOW_SPAN + 1):
        trial = best_pits.copy()
        trial[i] = p
        s = with_pits(comps, trial)
        if feasible(s):
            xs.append(p)
            ys.append(monte_carlo(s, S).mean())
    if not xs:
        print(f'Stop {i + 1}: no feasible alternative laps')
        continue
    xs, ys = np.array(xs), np.array(ys) - min(ys)
    window = xs[ys <= WINDOW_TOL]
    line, = ax.plot(xs, ys, marker='o', label=f'Stop {i + 1}')
    ax.axvspan(window.min(), window.max(), color=line.get_color(), alpha=0.15)
    print(f'Stop {i + 1}: ideal lap {xs[ys.argmin()]} | window L{window.min()}-L{window.max()}')

ax.set(xlabel='Pit lap', ylabel='Expected time lost (s)',
       title=f"Pit windows: {top.loc[0, 'label']} - {RACE_TITLE}")
ax.legend()
plt.show()

# 6. Pit wall
### 6.1 Dry pit-wall call: box now or stay out?
`pit_wall_call(lap, compound, tyre_age, used, under_sc)` searches every way to finish the race (up to 2 more stops) and compares boxing now with staying out. Try your own situations at the bottom of the cell.

In [ ]:
NEED_COMPOUNDS = 2 if TWO_COMPOUND_RULE else 1

def finish(R, c, age, used, stops):
    best = (np.inf, [])
    if len(used) >= NEED_COMPOUNDS and age + R <= MAX_STINT[c]:
        best = (stint_time(c, R, age), [(c, R)])
    if stops > 0:
        for k in range(1, R):
            if age + k > MAX_STINT[c]:
                break
            for new in AVAILABLE:
                t, plan = finish(R - k, new, 0, used | {new}, stops - 1)
                t += stint_time(c, k, age) + PIT_LOSS
                if t < best[0]:
                    best = (t, [(c, k)] + plan)
    return best

def describe(plan, lap):
    out = []
    for c, n in plan:
        lap += n
        out.append(f'{c[0]} to L{lap}')
    return ', '.join(out) or 'no legal plan'

def pit_wall_call(lap, compound, tyre_age, used=(), under_sc=False, max_stops=2):
    if compound not in AVAILABLE:
        print(f'No pace model for {compound}. Pick one of {AVAILABLE}.')
        return
    used = set(used) | {compound}
    R = TOTAL_LAPS - lap
    box_cost = PIT_LOSS * (SC_PIT_FACTOR if under_sc else 1.0)

    box = (np.inf, [])
    for new in AVAILABLE:
        t, plan = finish(R, new, 0, used | {new}, max_stops - 1)
        if t + box_cost < box[0]:
            box = (t + box_cost, plan)
    stay = finish(R, compound, tyre_age, used, max_stops)

    best = min(box[0], stay[0])
    print(f"Lap {lap}/{TOTAL_LAPS} | {compound} age {tyre_age} | {'SAFETY CAR' if under_sc else 'green flag'}")
    if not np.isfinite(best):
        print('   no legal way to finish from here\n')
        return
    for name, (t, plan) in [('BOX NOW ', box), ('STAY OUT', stay)]:
        gap = f'{t - best:+6.1f} s' if np.isfinite(t) else '   n/a'
        print(f'   {name}: {describe(plan, lap):32s} {gap}')
    print('   >>> CALL:', 'BOX BOX BOX' if box[0] <= stay[0] else 'STAY OUT', '\n')

# example calls built around the recommended plan
first, stop1 = BEST[0][0], best_pits[0]
early = max(MIN_STINT, stop1 - 4)
pit_wall_call(lap=early, compound=first, tyre_age=early)
pit_wall_call(lap=early, compound=first, tyre_age=early, under_sc=True)
pit_wall_call(lap=max(MIN_STINT, stop1 // 2), compound=first, tyre_age=max(MIN_STINT, stop1 // 2), under_sc=True)

### 6.2 Wet tyre crossovers and the rain call
With the default `GRIP` numbers, slicks are quickest below about 0.21 wetness, intermediates from 0.21 to 0.5, and full wets above that. `rain_call(lap, tyre, wetness, under_sc)` decides whether a tyre change pays for itself over the next `LOOKAHEAD` laps.

In [ ]:
w = np.linspace(0, 1, 101)
fig, ax = plt.subplots(figsize=(9, 4))
for k, col in enumerate(['#DA291C', COLORS['INTERMEDIATE'], COLORS['WET']]):
    ax.plot(w, (mult(k, w) - 1) * 100, lw=3, color=col, label=TYPES[k])
ax.set(xlabel='Track wetness (0 = dry, 1 = soaked)', ylabel='% slower than dry pace',
       title='Tyre crossover points')
ax.legend()
plt.show()

def rain_call(lap, tyre, wetness, under_sc=False):
    lap  = int(min(max(lap, 1), TOTAL_LAPS))
    left = TOTAL_LAPS - lap
    ref  = DRY_REF + FUEL[lap - 1]
    pace = [ref * mult(k, wetness) for k in range(3)]
    k, b = kind(tyre), int(np.argmin(pace))
    cost = PIT_LOSS * (SC_PIT_FACTOR if under_sc else 1.0)
    gain = (pace[k] - pace[b]) * min(left, LOOKAHEAD)
    print(f"Lap {lap}/{TOTAL_LAPS} | on {tyre} | wetness {wetness:.2f} | {'SAFETY CAR' if under_sc else 'racing'}")
    for j in range(3):
        print(f'   {TYPES[j]:13s} {pace[j]:7.2f} s/lap')
    if b != k and gain > cost:
        new = best_slick(left) if b == 0 else TYPES[b]
        print(f'   >>> CALL: BOX BOX BOX for {new}  (gain {gain:.1f} s > pit cost {cost:.1f} s)\n')
    else:
        print(f'   >>> CALL: STAY OUT  (gain {gain:.1f} s vs pit cost {cost:.1f} s)\n')

L = TOTAL_LAPS
rain_call(lap=round(L * 0.3), tyre=first, wetness=0.15)
rain_call(lap=round(L * 0.3), tyre=first, wetness=0.45)
rain_call(lap=round(L * 0.5), tyre='INTERMEDIATE', wetness=0.85)
rain_call(lap=round(L * 0.75), tyre='INTERMEDIATE', wetness=0.0)

# 7. Real world
### 7.1 Real strategies from the OpenF1 API
Finds the OpenF1 race session closest to the race date, then loads every driver's stints and pit stops. OpenF1 starts in 2023, so earlier races skip this part.

In [ ]:
stints = None
if DATA_YEAR < 2023:
    print(f'OpenF1 data starts in 2023, so there are no real stints to compare for {DATA_YEAR}.')
else:
    try:
        races = openf1('sessions', year=DATA_YEAR, session_name='Race')
        target = pd.Timestamp(race.date).tz_localize(None)
        SESSION_KEY = int(races.loc[(utc(races['date_start']) - target).abs().idxmin(), 'session_key'])

        stints  = openf1('stints',  session_key=SESSION_KEY)
        drivers = openf1('drivers', session_key=SESSION_KEY)
        pit_df  = openf1('pit',     session_key=SESSION_KEY)

        drivers = drivers[['driver_number', 'name_acronym']].drop_duplicates('driver_number')
        stints  = stints.merge(drivers, on='driver_number')
        stints['lap_end'] = stints['lap_end'].fillna(TOTAL_LAPS)

        lane = pit_df.filter(['pit_duration', 'lane_duration']).apply(pd.to_numeric, errors='coerce').max(axis=1)
        print(f'OpenF1 session {SESSION_KEY} | {len(stints)} stints | median pit-lane time {lane.median():.1f} s'
              f' | modelled pit loss {PIT_LOSS:.1f} s')
    except Exception as err:
        stints = None
        print(f'OpenF1 is not reachable right now ({err}). Run this cell again later.')

### 7.2 Real strategies vs the simulator's pick
Stint bars for each driver in finishing order, with the simulator's recommended plan as the hatched `SIM` row on top.

In [ ]:
if stints is None or stints.empty:
    print('No real stints to plot (see 7.1).')
else:
    order = race.results.sort_values('Position')['Abbreviation'].tolist()
    fig, ax = plt.subplots(figsize=(12, 8))
    for _, s in stints.iterrows():
        if s['name_acronym'] in order:
            ax.barh(order.index(s['name_acronym']), s['lap_end'] - s['lap_start'] + 1,
                    left=s['lap_start'] - 1, color=COLORS.get(s['compound'], 'grey'), edgecolor='black')
    left = 0
    for c, n in BEST:
        ax.barh(-1.5, n, left=left, color=COLORS[c], edgecolor='black', hatch='//')
        left += n
    ax.set_yticks([-1.5] + list(range(len(order))), ['SIM'] + order)
    ax.invert_yaxis()
    ax.set(xlabel='Lap', title=f'Real {DATA_YEAR} strategies vs simulator pick for {RACE_TITLE}')
    plt.show()

# 8. Race winner prediction

Race winner predictions are on the project website. Once qualifying is over, the site reads that weekend's practice and qualifying laps plus past results, gives every driver a win chance, explains the pick, and lets you simulate the race yourself.

### Find out the predicted race winner: **https://neevoswal19-maker.github.io/f1-strategy-suite/**

# Settings you can change

Your race, driver and cache choices are in the form in **1.1**. Every other setting is in **1.3**.

| Setting | What it does |
|---|---|
| `N_SIMS` | Simulated races per strategy. More = smoother results, slower |
| `FUEL_K` | Seconds per lap gained as fuel burns off (default 0.06) |
| `MIN_COMPOUND_LAPS` | Clean laps a compound needs before its pace curve is used |
| `MIN_STINT`, `MAX_STOPS` | Shortest allowed stint and most stops to consider |
| `DEFAULT_PIT_LOSS` | Pit loss used when a race has too few clean stops to measure it |
| `P_SC` | Chance of at least one safety car. Raise to ~0.8 for street circuits |
| `SC_LAPS`, `SC_SLOWDOWN` | How long each safety car lasts and how slow its laps are |
| `SC_PIT_FACTOR` | Share of the normal pit loss paid when stopping under a safety car |
| `P_SLOW_STOP`, `SLOW_STOP_S` | Chance and cost of a slow pit stop |
| `DEG_UNCERT` | Uncertainty in tyre wear (0.15 = ±15%) |
| `RAIN_DEFAULT` | What-if rain chance when there is no forecast (default 15%) |
| `WET_RACE_SHARE` | Above this share of wet-tyre laps, slick pace comes from another race |
| `GRIP` | How much slower each tyre type gets as the track gets wetter |
| `DRYING_LAPS` | Laps the track takes to dry after rain stops |
| `LOOKAHEAD` | How many laps ahead the team looks when making a weather call |
| `WINDOW_TOL`, `WINDOW_SPAN` | Pit window tolerance and how far around the ideal lap to test |
| `LOOKBACK_SEASONS` | Seasons searched for an earlier race at the same circuit |
| `pit_wall_call(...)` (6.1) | Test any lap, tyre and tyre age, with or without a safety car |
| `rain_call(...)` (6.2) | Test any lap, tyre and track wetness |

# Limitations

- Each strategy is raced by one car on an empty track: no traffic, undercuts, overcuts or track position.
- Dry tyre pace comes from one race, with one curve per compound. The fuel effect is a fixed estimate.
- Every stint starts on new tyres, even though real cars often start on tyres used in qualifying.
- Safety cars are simplified: fixed length, same lap time, random timing.
- Wet tyre pace (`GRIP`) is an assumption, not learned from data.
- For a past wet race, rain arrives at random laps, not when it really did.
- The simulated team reacts only to current conditions and doesn't know when the rain will stop.
- Upcoming races are projected from the last race at the same circuit. 2026 cars run under new rules, so lap times from earlier years don't carry over exactly. Stop counts and pit windows are still a fair guide.
- A brand-new circuit has no earlier race, so the notebook simulates the latest completed race instead.
- Forecasts only reach 16 days ahead, and "chance of rain" is not the same as rain that affects the race.
- FastF1 lap data starts in 2018 and OpenF1 in 2023.